In [ ]:
@file:DependsOn("org.postgresql:postgresql:42.7.4")
@file:OptIn(ExperimentalUuidApi::class)

import kotlin.uuid.ExperimentalUuidApi





In [ ]:
%use dataframe
%use kandy

In [ ]:
import bps.jdbc.getConfigFromResource

val (jdbcConfig, hikariConfig) = getConfigFromResource("/home/benji/.config/bps-budget-server/budget-server.yml")

In [ ]:
hikariConfig

In [ ]:
import bps.jdbc.configureDataSource

val dataSource = configureDataSource(jdbcConfig, hikariConfig)

In [ ]:
import java.util.UUID

val budgetAccessId = "7e8366d3-6710-4342-a3ea-26282ddb65f6"
val budgetName = "Benji's Budget"
val userId = UUID.fromString("c2d15fe2-3dfe-4517-b9ce-ab1b55f6b320")

In [ ]:
import bps.jdbc.JdbcFixture.Companion.transactOrThrow
import java.sql.PreparedStatement
import java.sql.ResultSet

val basicData = DataFrame.readSqlQuery(
    dataSource = dataSource,
//                                join users u on u.id = ba.user_id
    sqlQuery =     """
                            select b.general_account_name, b.id as budget_id, ba.time_zone, ba.budget_name, ba.analytics_start, a.id as general_account_id
                            from budgets b
                                join budget_access ba on b.id = ba.budget_id
                                join accounts a
                                    on a.name = b.general_account_name
                                    and a.type = 'category'
                                    and a.budget_id = b.id
                            where ba.budget_name = ?
                                and ba.user_id = ?
                        """.trimIndent(),
    limit = 1000,
) { statement: PreparedStatement ->
    statement.setString(1, budgetName)
    statement.setObject(2, userId, java.sql.Types.OTHER)
}

In [ ]:
basicData.schema()

In [ ]:
basicData

In [ ]:
@file:OptIn(ExperimentalUuidApi::class, ExperimentalTime::class)

import bps.jdbc.JdbcFixture
import kotlin.time.ExperimentalTime
import kotlin.time.toKotlinInstant
import kotlin.uuid.ExperimentalUuidApi
import kotlin.uuid.Uuid

val sqlQuery = """
                |select t.timestamp_utc, i.amount
                |from transaction_items i
                |join transactions t
                |    on i.transaction_id = t.id
                |    and i.budget_id = t.budget_id
                |join accounts a
                |    on i.account_id = a.id
                |    and i.budget_id = a.budget_id
                |where t.type = 'expense'
                |  and i.amount < 0
                |  and a.type = 'category'
                |  and t.timestamp_utc >= ?
                |  and t.timestamp_utc < ?
                |  and i.budget_id = ?
                |order by t.timestamp_utc asc
            """.trimMargin()
val spending = DataFrame.readSqlQuery(dataSource, sqlQuery) {
   with (JdbcFixture) {
       it.setInstant(1, basicData.analytics_start.values.first().toInstant().toKotlinInstant())
       it.setInstant(2, kotlin.time.Clock.System.now())
       it.setUuid(3, Uuid.parse(basicData.budget_id.values.first().toString()))
   }
}
spending.schema()

In [ ]:
spending

In [ ]:
@file:OptIn(ExperimentalTime::class)

import kotlin.time.ExperimentalTime

val groupedByDay =
    spending
        .convert { timestamp_utc }
        .with {
            it
                .toInstant()
                .toKotlinInstant()
                .toLocalDateTime(kotlinx.datetime.TimeZone.of("America/Chicago"))
                .date
        }
        .groupBy { timestamp_utc }
val spendingByDay =
    dataFrameOf(
        "date" to groupedByDay.keys.timestamp_utc.values().toList(),
        "amount" to groupedByDay.groups.map { it.sumOf { it.amount.toDouble() } }.toList()
    )
spendingByDay.describe()

In [ ]:
spendingByDay.plot {
    line {
        x(date)
        y(amount)
    }
    layout.size = 1600 to 500
}

In [ ]:
import kotlin.time.Duration

val spendingWithMa30 = spending.add("ma30") {
    if (basicData.analytics_start.values().first().toInstant().toKotlinInstant() + 30.days >
        timestamp_utc.toInstant().toKotlinInstant()
    )
        null
    else {
        relative(-(window - 1)..0).close.mean()
    }
}